<a href="https://colab.research.google.com/github/thanhnhan959604/MusicEmotionProject/blob/main/musiccnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa scipy -q

from google.colab import drive
drive.mount('/content/drive')

import os, re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, accuracy_score
from scipy.ndimage import zoom
from tqdm import tqdm
import librosa

print("Import xong")

Mounted at /content/drive
Import xong


In [ ]:
BASE_DIR   = '/content/drive/MyDrive/MusicEmotionProject'
CSV_PATH   = f'{BASE_DIR}/step7_train_ready.csv'   # ← data mới 4031 bài
AUDIO_DIR  = f'{BASE_DIR}/audio_previews'
MODEL_PATH = f'{BASE_DIR}/TAMS_MER_v2.pth'
CACHE_PATH = f'{BASE_DIR}/tams_cache_v2.pt'        # cache riêng cho data mới
LYRICS_COL = 'Lyrics'

# ── Audio ──────────────────────────────────────
SR       = 22050
DURATION = 30
N_MELS   = 128
TARGET_T = 256
HOP_LIST = [256, 512, 1024]

# ── Text ───────────────────────────────────────
MAX_SYLLABLES  = 200
VOCAB_MIN_FREQ = 2
N_TONES        = 7

# ── Train ──────────────────────────────────────
BATCH_SIZE   = 64
NUM_WORKERS  = 2
PIN_MEMORY   = True
K_FOLDS      = 5
EPOCHS_FOLD  = 80
EPOCHS_FINAL = 80
LR           = 2e-4
WEIGHT_DECAY = 1e-3
DROPOUT      = 0.4
EMBED_DIM    = 256
MIXUP_ALPHA  = 0.4
SMOOTH_EPS   = 0.05
PATIENCE_MAX = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị: {device.type.upper()}")
print(f"Data mới : {CSV_PATH}")

Thiết bị: CUDA
Data mới : /content/drive/MyDrive/MusicEmotionProject/step7_train_ready.csv


In [ ]:
# ══════════════════════════════════════════════
# AUDIO: Multi-resolution Mel Spectrogram
# ══════════════════════════════════════════════
def compute_multiresolution_mel(audio_path: str) -> np.ndarray:
    try:
        y, _ = librosa.load(audio_path, sr=SR, duration=DURATION, mono=True)
    except Exception:
        return np.zeros((3, N_MELS, TARGET_T), dtype=np.float32)

    channels = []
    for hop in HOP_LIST:
        mel    = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS, hop_length=hop,
            n_fft=max(1024, hop * 4)
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        T      = mel_db.shape[1]
        mel_db = zoom(mel_db, (1, TARGET_T / T), order=1) if T != TARGET_T else mel_db
        mel_db = mel_db[:, :TARGET_T]
        lo, hi = mel_db.min(), mel_db.max()
        mel_db = (mel_db - lo) / (hi - lo + 1e-6)
        channels.append(mel_db.astype(np.float32))

    return np.stack(channels, axis=0)   # (3, 128, 256)

# ══════════════════════════════════════════════
# TEXT: Tone-Aware Tokenizer (đặc thù tiếng Việt)
# ══════════════════════════════════════════════
_TONE_SETS = [
    set(),
    set('àằầèềìòồờùừỳ'),   # huyền
    set('áắấéếíóốớúứý'),   # sắc
    set('ảẳẩẻểỉỏổởủửỷ'),   # hỏi
    set('ãẵẫẽĩõỗỡũữỹ'),    # ngã
    set('ạặậẹệịọộợụựỵ'),   # nặng
]

def detect_tone(syllable: str) -> int:
    for ch in syllable.lower():
        for tone_id, tone_set in enumerate(_TONE_SETS[1:], start=1):
            if ch in tone_set:
                return tone_id
    return 0

def preprocess_text(text: str):
    text  = text.lower()
    text  = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    syls  = text.split()[:MAX_SYLLABLES]
    tones = [detect_tone(s) + 1 for s in syls]
    return syls, tones

def build_vocab(df, col=LYRICS_COL, min_freq=VOCAB_MIN_FREQ):
    freq = {}
    for txt in df[col].dropna():
        for tok in preprocess_text(str(txt))[0]:
            freq[tok] = freq.get(tok, 0) + 1
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, c in freq.items():
        if c >= min_freq:
            vocab[w] = len(vocab)
    print(f"Vocab size: {len(vocab):,} âm tiết")
    return vocab

def encode_syllables(syls, tones, vocab):
    pad = MAX_SYLLABLES - len(syls)
    return (
        [vocab.get(s, vocab['<UNK>']) for s in syls] + [0] * pad,
        tones + [0] * pad
    )

# ── Load CSV + build vocab ──────────────────
df_full    = pd.read_csv(CSV_PATH)
print(f"Tổng bài: {len(df_full):,}")

# kiểm tra cột cần thiết
for col in ['Spotify_ID', 'valence', 'energy', LYRICS_COL]:
    assert col in df_full.columns, f"Thiếu cột '{col}' trong CSV!"

VOCAB      = build_vocab(df_full)
VOCAB_SIZE = len(VOCAB)

Tổng bài: 4,031
Vocab size: 5,745 âm tiết


In [ ]:
def build_cache(df, audio_dir, vocab, lyrics_col=LYRICS_COL):
    mels, syls_list, tones_list, labels_list = [], [], [], []
    skipped_audio = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building cache"):
        tid      = str(row['Spotify_ID'])
        val, eng = float(row['valence']), float(row['energy'])

        audio_path = None
        for ext in ('.mp3', '.wav', '.ogg', '.flac'):
            p = os.path.join(audio_dir, f"{tid}{ext}")
            if os.path.exists(p):
                audio_path = p
                break
        if audio_path is None:
            skipped_audio += 1
            continue

        mel               = compute_multiresolution_mel(audio_path)
        lyrics            = str(row.get(lyrics_col, ''))
        syls, tones       = preprocess_text(lyrics)
        syl_ids, tone_ids = encode_syllables(syls, tones, vocab)

        mels.append(torch.from_numpy(mel))
        syls_list.append(torch.tensor(syl_ids,  dtype=torch.long))
        tones_list.append(torch.tensor(tone_ids, dtype=torch.long))
        labels_list.append(torch.tensor([val, eng], dtype=torch.float32))

    cache = {
        'mel':    torch.stack(mels),
        'syls':   torch.stack(syls_list),
        'tones':  torch.stack(tones_list),
        'labels': torch.stack(labels_list),
    }
    torch.save(cache, CACHE_PATH)
    print(f"\nCache lưu: {CACHE_PATH}")
    print(f"   Hợp lệ  : {len(mels):,} bài")
    print(f"   Bỏ qua  : {skipped_audio} (không có audio)")
    print(f"   Dung lượng: {os.path.getsize(CACHE_PATH)/1e6:.1f} MB")
    return cache

# ── Load hoặc build ─────────────────────────
if not os.path.exists(CACHE_PATH):
    print("Chưa có cache → đang build (~15-20 phút)...")
    cache = build_cache(df_full, AUDIO_DIR, VOCAB)
else:
    print("Cache đã có → load...")
    cache = torch.load(CACHE_PATH, weights_only=False)
    print(f"   Mẫu: {len(cache['mel']):,} | Size: {os.path.getsize(CACHE_PATH)/1e6:.1f} MB")

cached_dataset = TensorDataset(
    cache['mel'], cache['syls'], cache['tones'], cache['labels']
)
print(f"\nDataset ready: {len(cached_dataset):,} mẫu")

Cache đã có → load...
   Mẫu: 4,031 | Size: 1598.0 MB

Dataset ready: 4,031 mẫu


In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.gap(x).view(b, c)).view(b, c, 1, 1)

class SEResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se   = SEBlock(out_ch)
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.act  = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.se(self.body(x)) + self.skip(x))

class TemporalAttentionPool(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Sequential(
            nn.Conv1d(channels, channels // 4, 1), nn.Tanh(),
            nn.Conv1d(channels // 4, 1, 1),
        )
    def forward(self, x):
        w = torch.softmax(self.score(x), dim=2)
        return (x * w).sum(dim=2)

class AudioCNN(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, dropout=DROPOUT):
        super().__init__()
        d = dropout * 0.5
        self.encoder = nn.Sequential(
            SEResBlock(3,   32,  dropout=d), nn.MaxPool2d(2),
            SEResBlock(32,  64,  dropout=d), nn.MaxPool2d(2),
            SEResBlock(64,  128, dropout=d), nn.MaxPool2d(2),
            SEResBlock(128, 256, dropout=d),
            nn.AdaptiveAvgPool2d((1, None))
        )
        self.tap  = TemporalAttentionPool(256)
        self.proj = nn.Sequential(
            nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.proj(self.tap(self.encoder(x).squeeze(2)))

class ToneAwareTextCNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, n_tones=N_TONES,
                 embed_dim=EMBED_DIM, num_filters=64,
                 kernel_sizes=(1, 2, 3, 4, 5), dropout=DROPOUT):
        super().__init__()
        SEM_DIM, TONE_DIM = 128, 16
        self.emb_semantic = nn.Embedding(vocab_size, SEM_DIM,  padding_idx=0)
        self.emb_tone     = nn.Embedding(n_tones,    TONE_DIM, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(SEM_DIM + TONE_DIM, num_filters,
                          kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm1d(num_filters), nn.ReLU(inplace=True),
            ) for k in kernel_sizes
        ])
        self.proj = nn.Sequential(
            nn.Linear(num_filters * len(kernel_sizes), embed_dim),
            nn.LayerNorm(embed_dim), nn.Dropout(dropout)
        )
    def forward(self, syl_ids, tone_ids):
        x = torch.cat([self.emb_semantic(syl_ids),
                        self.emb_tone(tone_ids)], dim=2).permute(0, 2, 1)
        return self.proj(torch.cat(
            [conv(x).max(dim=2).values for conv in self.convs], dim=1
        ))

class GatedCrossModalAttention(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_heads=4, dropout=DROPOUT):
        super().__init__()
        self.t2a    = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.a2t    = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_t = nn.LayerNorm(embed_dim)
        self.norm_a = nn.LayerNorm(embed_dim)
        self.gate_a = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
        self.gate_t = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())
    def forward(self, audio, text):
        a_s, t_s = audio.unsqueeze(1), text.unsqueeze(1)
        t2a = self.norm_t(self.t2a(t_s, a_s, a_s)[0] + t_s).squeeze(1)
        a2t = self.norm_a(self.a2t(a_s, t_s, t_s)[0] + a_s).squeeze(1)
        ctx = torch.cat([audio, text], dim=1)
        return audio + self.gate_a(ctx) * a2t, text + self.gate_t(ctx) * t2a

class TAMS_MER(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                 hidden_dim=256, dropout=DROPOUT):
        super().__init__()
        self.audio_cnn = AudioCNN(embed_dim, dropout)
        self.text_cnn  = ToneAwareTextCNN(vocab_size, embed_dim=embed_dim, dropout=dropout)
        self.fusion    = GatedCrossModalAttention(embed_dim, dropout=dropout)
        def head():
            return nn.Sequential(
                nn.Linear(embed_dim * 2, hidden_dim),
                nn.LayerNorm(hidden_dim), nn.ReLU(inplace=True), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2), nn.ReLU(inplace=True), nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1)
            )
        self.valence_head = head()
        self.energy_head  = head()
    def forward(self, mel, syl_ids, tone_ids):
        a_feat = self.audio_cnn(mel)
        t_feat = self.text_cnn(syl_ids, tone_ids)
        a_out, t_out = self.fusion(a_feat, t_feat)
        val = self.valence_head(torch.cat([t_out, a_out], dim=1))
        eng = self.energy_head(torch.cat([a_out, t_out], dim=1))
        return torch.sigmoid(torch.cat([val, eng], dim=1))

# kiểm tra
m = TAMS_MER().to(device)
total = sum(p.numel() for p in m.parameters())
print(f"Model OK — {total:,} tham số")
del m

Model OK — 3,399,155 tham số


In [ ]:
class SmoothedMSELoss(nn.Module):
    def __init__(self, eps=SMOOTH_EPS, l1_w=0.1):
        super().__init__()
        self.eps  = eps
        self.l1_w = l1_w
        self.mse  = nn.MSELoss()
        self.l1   = nn.L1Loss()
    def forward(self, pred, target):
        t = target * (1 - self.eps) + 0.5 * self.eps
        return self.mse(pred, t) + self.l1_w * self.l1(pred, t)

def compute_sample_weights(labels_tensor):
    """
    Tính weight cho từng sample dựa trên góc phần tư.
    Vui-Êm (294 bài) sẽ có weight ~5.5× so với Buồn-Êm (1606 bài).
    """
    v    = (labels_tensor[:, 0] >= 0.5).long()
    e    = (labels_tensor[:, 1] >= 0.5).long()
    quad = v * 2 + e   # 0=Buồn-Êm  1=Buồn-Sôi  2=Vui-Êm  3=Vui-Sôi

    counts = torch.bincount(quad, minlength=4).float()
    total  = len(labels_tensor)

    names = ['Buồn-Êm ', 'Buồn-Sôi', 'Vui-Êm  ', 'Vui-Sôi ']
    print("\nPhân phối & weight bù:")
    print(f"  {'Góc':12s}  {'Bài':>6s}  {'%':>6s}  {'Weight':>8s}")
    print("  " + "─"*40)
    for i, (n, c) in enumerate(zip(names, counts)):
        w = total / (4.0 * c)
        print(f"  {n}  {int(c):>6,}  {c/total*100:>5.1f}%  {w:>8.2f}×")

    # weight mỗi sample
    sample_w = torch.zeros(total)
    for i in range(total):
        sample_w[i] = total / (4.0 * counts[quad[i]])
    return sample_w, quad.numpy()

def mixup_audio(mel, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return mel, labels
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(mel.size(0)).to(mel.device)
    return (lam * mel + (1 - lam) * mel[idx],
            lam * labels + (1 - lam) * labels[idx])

# ── Tính weights một lần, dùng cho cả KFold + Final ──
sample_weights, quad_labels = compute_sample_weights(cache['labels'])
print(f"\nWeights: min={sample_weights.min():.2f}  max={sample_weights.max():.2f}")


Phân phối & weight bù:
  Góc              Bài       %    Weight
  ────────────────────────────────────────
  Buồn-Êm    1,606   39.8%      0.63×
  Buồn-Sôi   1,054   26.1%      0.96×
  Vui-Êm       294    7.3%      3.43×
  Vui-Sôi    1,077   26.7%      0.94×

Weights: min=0.63  max=3.43


In [ ]:
def make_balanced_loader(dataset, indices, weights_all,
                         batch_size, drop_last=True):
    """Train loader: WeightedRandomSampler bù mất cân bằng."""
    sampler = WeightedRandomSampler(
        weights=weights_all[indices],
        num_samples=len(indices),
        replacement=True
    )
    return DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size, sampler=sampler,
        drop_last=drop_last, num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY, persistent_workers=True
    )

def make_eval_loader(dataset, indices, batch_size):
    """Test/val loader: KHÔNG weighted — đánh giá phân phối thực."""
    return DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=True
    )

def evaluate_kfold(dataset, sample_weights, quad_labels):
    print(f"\n{'='*58}")
    print("  STRATIFIED K-FOLD + WEIGHTED SAMPLER — TAMS-MER v2")
    print(f"{'='*58}")

    skfold  = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    results = {k: [] for k in ['r2_v','r2_e','acc_v','acc_e','acc_4q']}

    for fold, (tr_ids, te_ids) in enumerate(
            skfold.split(np.zeros(len(dataset)), quad_labels)):

        print(f"\n{'─'*22} FOLD {fold+1}/{K_FOLDS} {'─'*22}")

        tr_loader = make_balanced_loader(dataset, tr_ids, sample_weights, BATCH_SIZE)
        te_loader = make_eval_loader(dataset, te_ids, BATCH_SIZE)

        model     = TAMS_MER().to(device)
        criterion = SmoothedMSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

        best_loss, best_state, patience = float('inf'), None, 0

        for ep in range(EPOCHS_FOLD):
            # ── train ──────────────────────────────
            model.train()
            tr_loss = 0.0
            for mel, syls, tones, lbl in tr_loader:
                mel, syls, tones, lbl = (mel.to(device), syls.to(device),
                                          tones.to(device), lbl.to(device))
                mel_m, lbl_m = mixup_audio(mel, lbl)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(mel_m, syls, tones), lbl_m)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                tr_loss += loss.item()
            scheduler.step()

            # ── val ────────────────────────────────
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for mel, syls, tones, lbl in te_loader:
                    val_loss += criterion(
                        model(mel.to(device), syls.to(device), tones.to(device)),
                        lbl.to(device)
                    ).item()

            tr_loss  /= len(tr_loader)
            val_loss /= len(te_loader)

            if val_loss < best_loss:
                best_loss  = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience   = 0
            else:
                patience += 1
                if patience >= PATIENCE_MAX:
                    print(f"  Early stop epoch {ep+1}")
                    break

            if (ep + 1) % 10 == 0:
                print(f"  Ep {ep+1:02d} | train={tr_loss:.4f}  val={val_loss:.4f}  "
                      f"patience={patience}")

        # ── evaluate ───────────────────────────────
        model.load_state_dict(best_state)
        model.eval()
        preds_all, lbls_all = [], []
        with torch.no_grad():
            for mel, syls, tones, lbl in te_loader:
                out = model(mel.to(device), syls.to(device), tones.to(device))
                preds_all.append(out.cpu().numpy())
                lbls_all.append(lbl.numpy())

        P = np.vstack(preds_all)
        L = np.vstack(lbls_all)

        r2_v  = r2_score(L[:,0], P[:,0])
        r2_e  = r2_score(L[:,1], P[:,1])
        acc_v = accuracy_score((L[:,0]>=.5).astype(int),
                                (P[:,0]>=.5).astype(int)) * 100
        acc_e = accuracy_score((L[:,1]>=.5).astype(int),
                                (P[:,1]>=.5).astype(int)) * 100
        pred_q = (P[:,0]>=.5).astype(int)*2 + (P[:,1]>=.5).astype(int)
        true_q = (L[:,0]>=.5).astype(int)*2 + (L[:,1]>=.5).astype(int)
        acc_4  = accuracy_score(true_q, pred_q) * 100

        # per-class — theo dõi Vui-Êm (góc 2)
        quad_names = ['Buồn-Êm ','Buồn-Sôi','Vui-Êm  ','Vui-Sôi ']
        print(f"\n  R2   → val: {r2_v:.4f}  eng: {r2_e:.4f}")
        print(f"  2-Cat → val: {acc_v:.1f}%  eng: {acc_e:.1f}%")
        print(f"  4-Cat overall: {acc_4:.1f}%")
        print(f"  Per-class:")
        for q in range(4):
            mask = (true_q == q)
            if mask.sum() > 0:
                c_acc = (pred_q[mask] == q).mean() * 100
                print(f"    {quad_names[q]}: {c_acc:.1f}%  ({mask.sum()} mẫu)")

        for k, val in zip(['r2_v','r2_e','acc_v','acc_e','acc_4q'],
                           [r2_v, r2_e, acc_v, acc_e, acc_4]):
            results[k].append(val)

    print("  KẾT QUẢ TRUNG BÌNH 5-FOLD — TAMS-MER v2")
    print(f"  [Valence] R2: {np.mean(results['r2_v']):.4f}  |  "
          f"2-Cat: {np.mean(results['acc_v']):.2f}%")
    print(f"  [Energy]  R2: {np.mean(results['r2_e']):.4f}  |  "
          f"2-Cat: {np.mean(results['acc_e']):.2f}%")
    print(f"  [4-Class] Accuracy: {np.mean(results['acc_4q']):.2f}%")
    return results

results = evaluate_kfold(cached_dataset, sample_weights, quad_labels)


  STRATIFIED K-FOLD + WEIGHTED SAMPLER — TAMS-MER v2

────────────────────── FOLD 1/5 ──────────────────────
  Ep 10 | train=0.0246  val=0.0280  patience=0
  Ep 20 | train=0.0223  val=0.0299  patience=4
  Ep 30 | train=0.0218  val=0.0290  patience=2
  Ep 40 | train=0.0179  val=0.0256  patience=2
  Ep 50 | train=0.0158  val=0.0250  patience=8
  Ep 60 | train=0.0148  val=0.0252  patience=6
  Ep 70 | train=0.0160  val=0.0254  patience=4
  Ep 80 | train=0.0145  val=0.0249  patience=14

  R2   → val: 0.5322  eng: 0.6601
  2-Cat → val: 79.7%  eng: 80.4%
  4-Cat overall: 64.4%
  Per-class:
    Buồn-Êm : 82.3%  (322 mẫu)
    Buồn-Sôi: 40.5%  (210 mẫu)
    Vui-Êm  : 39.0%  (59 mẫu)
    Vui-Sôi : 68.1%  (216 mẫu)

────────────────────── FOLD 2/5 ──────────────────────
  Ep 10 | train=0.0255  val=0.0269  patience=0
  Ep 20 | train=0.0227  val=0.0254  patience=3
  Ep 30 | train=0.0223  val=0.0256  patience=1
  Ep 40 | train=0.0186  val=0.0231  patience=1
  Ep 50 | train=0.0166  val=0.0242  patien

In [ ]:
def train_final(dataset, sample_weights):
    print(f" FINAL MODEL — {len(dataset):,} bài")

    all_ids   = list(range(len(dataset)))
    loader    = make_balanced_loader(dataset, all_ids, sample_weights,
                                     BATCH_SIZE, drop_last=True)
    model     = TAMS_MER().to(device)
    criterion = SmoothedMSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    for ep in range(EPOCHS_FINAL):
        model.train()
        tr_loss = 0.0
        for mel, syls, tones, lbl in loader:
            mel, syls, tones, lbl = (mel.to(device), syls.to(device),
                                      tones.to(device), lbl.to(device))
            mel_m, lbl_m = mixup_audio(mel, lbl)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(mel_m, syls, tones), lbl_m)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item()
        scheduler.step()
        if (ep + 1) % 10 == 0:
            print(f"  Epoch [{ep+1:02d}/{EPOCHS_FINAL}] loss={tr_loss/len(loader):.4f}")

    torch.save({
        'model_state': model.state_dict(),
        'vocab':       VOCAB,
        'config': {
            'vocab_size':    VOCAB_SIZE,
            'embed_dim':     EMBED_DIM,
            'n_mels':        N_MELS,
            'target_t':      TARGET_T,
            'max_syllables': MAX_SYLLABLES,
        }
    }, MODEL_PATH)
    print(f"\nĐã lưu model: {MODEL_PATH}")
    return model

final_model = train_final(cached_dataset, sample_weights)

 FINAL MODEL — 4,031 bài
  Epoch [10/80] loss=0.0254
  Epoch [20/80] loss=0.0230
  Epoch [30/80] loss=0.0217
  Epoch [40/80] loss=0.0175
  Epoch [50/80] loss=0.0172
  Epoch [60/80] loss=0.0145
  Epoch [70/80] loss=0.0169
  Epoch [80/80] loss=0.0148

Đã lưu model: /content/drive/MyDrive/MusicEmotionProject/TAMS_MER_v2.pth


In [ ]:
def predict(audio_path: str, lyrics: str, checkpoint=MODEL_PATH):
    ckpt  = torch.load(checkpoint, map_location=device, weights_only=False)
    cfg   = ckpt['config']
    vocab = ckpt['vocab']

    model = TAMS_MER(vocab_size=cfg['vocab_size'],
                     embed_dim=cfg['embed_dim']).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    mel    = torch.from_numpy(
                 compute_multiresolution_mel(audio_path)
             ).unsqueeze(0).to(device)

    syls, tones       = preprocess_text(lyrics)
    syl_ids, tone_ids = encode_syllables(syls, tones, vocab)
    syl_t  = torch.tensor(syl_ids,  dtype=torch.long).unsqueeze(0).to(device)
    tone_t = torch.tensor(tone_ids, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(mel, syl_t, tone_t).cpu().numpy()[0]

    v, e = float(pred[0]), float(pred[1])
    quad = {
        (1,1): "Vui – Sôi động  (Happy / Energetic)",
        (1,0): "Vui – Êm dịu   (Happy / Calm)",
        (0,1): "Căng – Sôi động (Tense / Angry)",
        (0,0): "Buồn – Êm dịu  (Sad / Calm)",
    }[(int(v >= .5), int(e >= .5))]

    print(f"Valence : {v:.4f}")
    print(f"Energy  : {e:.4f}")
    print(f"Cảm xúc : {quad}")
    return v, e

# ── Demo ────────────────────────────────────
predict(
    f'{BASE_DIR}/hatMaiKhucQuanHanh.mp3',
    "Đời mình là một khúc quân hành, đời mình là bài ca chiến sĩ. Ta ca vang, triền miên qua tháng ngày, lượn bay trên núi đồi biên cương đến nơi đảo xa. Mãi mãi lòng chúng ta, ca bài ca người lính. Mãi mãi lòng chúng ta, vẫn hát khúc quân hành ca. Dù rằng đời ta thích hoa hồng, kẻ thù buộc ta ôm cây súng. Ta yêu sao làng quê non nước mình. Tình quê hương vút thành thanh âm khúc quân hành ca. Mãi mãi lòng chúng ta, ca bài ca người lính. Mãi mãi lòng chúng ta, vẫn hát khúc quân hành ca."
)

Valence : 0.7943
Energy  : 0.5044
Cảm xúc : Vui – Sôi động  (Happy / Energetic)


(0.7943009734153748, 0.5044368505477905)